# M05E · Web Scraping & Biological Databases Homework: 6 Capstone Tasks
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M05_Databases_and_Scraping/M05E_Web_Scraping_Homework.ipynb)

⏱ ~180 min · Difficulty: ★★★ · Prerequisites: M04B (Matplotlib), M04C (Seaborn), M05A (Entrez), M05B (UniProt/PDB/KEGG), M05D (Web Scraping)


## 1. Learning Outcomes

By the end of this notebook, you will be able to:

- **Synthesize**: Integrate web scraping, REST APIs, and database querying into 6 end-to-end biological data extraction pipelines.
- **Formulate**: Author formal 3-line Ethics Statements for each data-harvesting workflow specifying robots.txt verification, rate limiting, and attribution.
- **Process**: Transform raw, semi-structured, and tabular online records into standardized, publication-ready CSV files.
- **Visualize**: Generate publication-grade analytical plots (distribution histograms, phase distributions, multi-panel bars, trendlines) using Matplotlib and Seaborn.
- **Validate**: Implement rigorous automated assertions and offline regression safeguards ensuring data pipelines remain reproducible.

## 2. Prerequisites


Before starting this homework assignment, you should have completed:
- Plot generation and publication styling with Matplotlib and Seaborn ([`M04B`](file:///modules/M04_Visualisation/M04B_Matplotlib.ipynb), [`M04C`](file:///modules/M04_Visualisation/M04C_Seaborn_and_Styling.ipynb)).
- Querying NCBI Entrez, UniProt, PDB, and KEGG ([`M05A`](file:///modules/M05_Databases_and_Scraping/M05A_Entrez_GenBank_PubMed.ipynb), [`M05B`](file:///modules/M05_Databases_and_Scraping/M05B_UniProt_PDB_KEGG.ipynb)).
- Secrets management and polite rate-limiting architectures ([`M05C`](file:///modules/M05_Databases_and_Scraping/M05C_Best_Practices_API_Keys.ipynb)).
- BeautifulSoup DOM parsing and HTML traversal ([`M05D`](file:///modules/M05_Databases_and_Scraping/M05D_Web_Scraping_Fundamentals.ipynb)).


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install biopython seaborn requests beautifulsoup4 lxml python-dotenv
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    # Resolve repository root robustly from modules/ or repo root
    p = Path.cwd()
    while p != p.parent and not (p / "COURSE_PLAN.md").exists():
        p = p.parent
    ROOT = p if (p / "COURSE_PLAN.md").exists() else Path.cwd().resolve()

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "M05E"
OUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests, re, time, io
from bs4 import BeautifulSoup
from Bio import Entrez, SeqIO, Medline
import seaborn as sns

print(f"Python: {sys.version.split()[0]}")
print(f"Working Directory: {Path.cwd()}")
print(f"DATA_DIR: {DATA_DIR} (exists: {DATA_DIR.exists()})")
print(f"OUT_DIR:  {OUT_DIR} (ready)")

## Homework Architecture & Submission Instructions

This homework assignment comprises **6 structured real-world computational biology data harvesting tasks**.

### Mandatory Deliverables for Each Task (HW1 – HW6):
1. **Python Implementation**: Resilient code that retrieves, parses, and cleans the target biological records.
2. **Output CSV Table**: Persisted in `outputs/M05E/` containing all required fields.
3. **Publication-Grade Figure**: Persisted in `outputs/M05E/` at 300 DPI visualizing the extracted data.
4. **Three-Line Ethics Statement**: A mandatory documentation block answering:
   - *Line 1 (Permission)*: Was `robots.txt` audited or an official API used?
   - *Line 2 (Politeness)*: What rate-limiting delay and User-Agent identification were applied?
   - *Line 3 (Attribution)*: How will the source data repository and accession release be cited?

### Practice Exercise HW1 (★★★) — Gene Expression Summary Pipeline

**Problem**: ### HW1: Gene Expression Profile Harvesting
**Biological Motivation**: Investigating the tissue-specific expression of prominent cancer biomarkers (e.g., `TP53`, `BRCA1`, `EGFR`, `MYC`, `PTEN`) reveals whether target genes function ubiquitously (housekeeping) or display organ-restricted activity.

- **Task Specification**:
  1. Extract expression values for at least 5 genes across at least 3 tissues (using `data/tables/expression_tissues.csv` or public repository).
  2. Compute mean expression and standard error (SE) per gene per tissue.
  3. Export tidy summary table to `outputs/M05E/gene_expression.csv`.
  4. Generate a grouped bar plot with error bars (`outputs/M05E/hw1_expression.png`) comparing gene expression across tissues.
  5. Include your 3-line Ethics Statement.


- **Input**: Path to expression table, output CSV, and output PNG
- **Expected Output**: Generated gene_expression.csv and hw1_expression.png

In [ ]:
# HW1: Gene Expression Profile Harvesting

# 1. Author your 3-line Ethics Statement:
# Line 1 (Permission): ...
# Line 2 (Politeness): ...
# Line 3 (Attribution): ...

def execute_hw1_pipeline(data_path: Path, out_csv: Path, out_img: Path) -> pd.DataFrame:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test HW1 — run me. No output means you passed.
out_csv_hw1 = OUT_DIR / "gene_expression.csv"
out_img_hw1 = OUT_DIR / "hw1_expression.png"
df_hw1 = execute_hw1_pipeline(DATA_DIR / "tables" / "expression_tissues.csv", out_csv_hw1, out_img_hw1)

assert out_csv_hw1.exists(), "HW1 CSV not generated"
assert out_img_hw1.exists(), "HW1 plot not generated"
assert "gene" in df_hw1.columns and "mean_tpm" in df_hw1.columns
assert len(df_hw1) >= 5, "Summary table must contain at least 5 gene-tissue rows"
print("HW1 passed ✔")

### Practice Exercise HW2 (★★★) — Protein Information Harvesting via UniProt

**Problem**: ### HW2: Protein Information Harvesting via UniProt
**Biological Motivation**: Comparing biochemical properties across protein families reveals evolutionary adaptations in molecular mass, subcellular localization, and domain architecture.

- **Task Specification**:
  1. Harvest protein information for 10 key human proteins: `['TP53', 'BRCA1', 'BRCA2', 'EGFR', 'PTEN', 'RB1', 'MYC', 'KRAS', 'ATM', 'CDK4']`.
  2. For each protein, retrieve: `Accession`, `Entry Name`, `Protein Name`, `Length (aa)`, `Organism`.
  3. Export the table to `outputs/M05E/protein_info.csv`.
  4. Generate a publication-quality histogram of protein length distribution (`outputs/M05E/hw2_protein_length.png`) with mean vertical line.
  5. Include your 3-line Ethics Statement.


- **Input**: List of 10 gene symbols, output CSV, and output PNG
- **Expected Output**: Generated protein_info.csv and hw2_protein_length.png

In [ ]:
# HW2: Protein Information Harvesting

# 1. Author your 3-line Ethics Statement:
# Line 1 (Permission): ...
# Line 2 (Politeness): ...
# Line 3 (Attribution): ...

def execute_hw2_pipeline(gene_list: list[str], out_csv: Path, out_img: Path) -> pd.DataFrame:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test HW2 — run me. No output means you passed.
out_csv_hw2 = OUT_DIR / "protein_info.csv"
out_img_hw2 = OUT_DIR / "hw2_protein_length.png"
genes_10 = ["TP53", "BRCA1", "BRCA2", "EGFR", "PTEN", "RB1", "MYC", "KRAS", "ATM", "CDK4"]
df_hw2 = execute_hw2_pipeline(genes_10, out_csv_hw2, out_img_hw2)

assert out_csv_hw2.exists()
assert out_img_hw2.exists()
assert len(df_hw2) == 10
assert "accession" in df_hw2.columns and "length" in df_hw2.columns
assert df_hw2["length"].min() > 100
print("HW2 passed ✔")

### Practice Exercise HW3 (★★★) — Clinical Trial Information Extraction

**Problem**: ### HW3: Clinical Trial Information Extraction
**Biological Motivation**: Tracking translational drug development programs (phases I–IV) reveals which therapeutic strategies are progressing toward FDA approval.

- **Task Specification**:
  1. Retrieve clinical trials evaluating targeted therapies (e.g. for p53, Breast Neoplasm, or Wilson Disease).
  2. Extract: `Trial_ID`, `Title`, `Phase`, `Status`, `Start_Year`.
  3. Export results to `outputs/M05E/clinical_trials.csv`.
  4. Generate a bar plot showing trial distribution across clinical phases (`outputs/M05E/hw3_clinical_phases.png`).
  5. Include your 3-line Ethics Statement.


- **Input**: Disease query string, output CSV, and output PNG
- **Expected Output**: Generated clinical_trials.csv and hw3_clinical_phases.png

In [ ]:
# HW3: Clinical Trial Information Extraction

# 1. Author your 3-line Ethics Statement:
# Line 1 (Permission): ...
# Line 2 (Politeness): ...
# Line 3 (Attribution): ...

def execute_hw3_pipeline(disease_query: str, out_csv: Path, out_img: Path) -> pd.DataFrame:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test HW3 — run me. No output means you passed.
out_csv_hw3 = OUT_DIR / "clinical_trials.csv"
out_img_hw3 = OUT_DIR / "hw3_clinical_phases.png"
df_hw3 = execute_hw3_pipeline("p53 cancer", out_csv_hw3, out_img_hw3)

assert out_csv_hw3.exists()
assert out_img_hw3.exists()
assert len(df_hw3) >= 5
assert "trial_id" in df_hw3.columns and "phase" in df_hw3.columns
print("HW3 passed ✔")

### Practice Exercise HW4 (★★★) — Taxonomic Lineage Extraction

**Problem**: ### HW4: Taxonomic Lineage Extraction
**Biological Motivation**: Mapping species to their evolutionary hierarchy (Kingdom, Phylum, Class, Order, Family, Genus) provides phylogenetic context for comparative genomics.

- **Task Specification**:
  1. Retrieve taxonomic classification for 10 model organisms:
     `['Homo sapiens', 'Mus musculus', 'Drosophila melanogaster', 'Caenorhabditis elegans', 'Danio rerio', 'Arabidopsis thaliana', 'Saccharomyces cerevisiae', 'Escherichia coli', 'Bacillus subtilis', 'Mycobacterium tuberculosis']`.
  2. Extract: `Scientific_Name`, `Superkingdom`, `Phylum`, `TaxId`.
  3. Export results to `outputs/M05E/taxonomy.csv`.
  4. Generate a bar chart showing organism distribution by Phylum or Superkingdom (`outputs/M05E/hw4_taxonomy.png`).
  5. Include your 3-line Ethics Statement.


- **Input**: List of 10 species names, output CSV, and output PNG
- **Expected Output**: Generated taxonomy.csv and hw4_taxonomy.png

In [ ]:
# HW4: Taxonomic Lineage Extraction

# 1. Author your 3-line Ethics Statement:
# Line 1 (Permission): ...
# Line 2 (Politeness): ...
# Line 3 (Attribution): ...

def execute_hw4_pipeline(species_list: list[str], out_csv: Path, out_img: Path) -> pd.DataFrame:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test HW4 — run me. No output means you passed.
out_csv_hw4 = OUT_DIR / "taxonomy.csv"
out_img_hw4 = OUT_DIR / "hw4_taxonomy.png"
species_10 = [
    "Homo sapiens", "Mus musculus", "Drosophila melanogaster", "Caenorhabditis elegans",
    "Danio rerio", "Arabidopsis thaliana", "Saccharomyces cerevisiae",
    "Escherichia coli", "Bacillus subtilis", "Mycobacterium tuberculosis"
]
df_hw4 = execute_hw4_pipeline(species_10, out_csv_hw4, out_img_hw4)

assert out_csv_hw4.exists()
assert out_img_hw4.exists()
assert len(df_hw4) == 10
assert "scientific_name" in df_hw4.columns and "phylum" in df_hw4.columns
assert "superkingdom" in df_hw4.columns
print("HW4 passed ✔")

### Practice Exercise HW5 (★★★) — Pathway Overlap Analysis

**Problem**: ### HW5: Biological Pathway Member Extraction & Overlap Analysis
**Biological Motivation**: Intracellular signaling networks operate as interconnected modules; genes shared across pathways frequently represent central hubs and therapeutic bottlenecks.

- **Task Specification**:
  1. Extract genes participating in 3 human KEGG pathways:
     - `hsa04115`: p53 signaling pathway
     - `hsa04110`: Cell cycle
     - `hsa04210`: Apoptosis
  2. Compute pairwise gene overlaps (intersections) and unique member counts.
  3. Export consolidated gene list to `outputs/M05E/pathway_genes.csv`.
  4. Generate a comparative bar plot showing total gene count and cross-pathway shared count (`outputs/M05E/hw5_pathways.png`).
  5. Include your 3-line Ethics Statement.


- **Input**: Dictionary of 3 KEGG pathway IDs, output CSV, and output PNG
- **Expected Output**: Generated pathway_genes.csv and hw5_pathways.png

In [ ]:
# HW5: Biological Pathway Member Extraction & Overlap Analysis

# 1. Author your 3-line Ethics Statement:
# Line 1 (Permission): ...
# Line 2 (Politeness): ...
# Line 3 (Attribution): ...

def execute_hw5_pipeline(pathways: dict[str, str], out_csv: Path, out_img: Path) -> pd.DataFrame:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test HW5 — run me. No output means you passed.
out_csv_hw5 = OUT_DIR / "pathway_genes.csv"
out_img_hw5 = OUT_DIR / "hw5_pathways.png"
pws = {
    "hsa04115": "p53 signaling pathway",
    "hsa04110": "Cell cycle",
    "hsa04210": "Apoptosis"
}
df_hw5 = execute_hw5_pipeline(pws, out_csv_hw5, out_img_hw5)

assert out_csv_hw5.exists()
assert out_img_hw5.exists()
assert len(df_hw5) >= 15
assert "pathway_id" in df_hw5.columns and "gene_symbol" in df_hw5.columns
# Confirm TP53 is present in all 3 pathways!
tp53_pws = df_hw5[df_hw5["gene_symbol"] == "TP53"]["pathway_id"].unique()
assert len(tp53_pws) == 3, "TP53 must participate in all three signaling pathways!"
print("HW5 passed ✔")

### Practice Exercise HW6 (★★★) — PubMed Literature Mining & Trend Analysis

**Problem**: ### HW6: PubMed Literature Mining & Publication Trend
**Biological Motivation**: Quantifying scientific publication velocity identifies emerging breakthrough technologies (e.g. CRISPR genome editing) and tracks the expansion of biomedical knowledge.

- **Task Specification**:
  1. Search PubMed for at least 25 citations on a focused biomedical topic (e.g., `'CRISPR Cas9 gene editing'` or `'cuproptosis'`).
  2. For each article, extract: `PMID`, `Title`, `Journal`, `Year`.
  3. Export to `outputs/M05E/pubmed_abstracts.csv`.
  4. Generate a publication trend bar plot (`outputs/M05E/hw6_pubmed_trend.png`) displaying annual paper counts.
  5. Include your 3-line Ethics Statement.


- **Input**: Query string, max records integer, output CSV, and output PNG
- **Expected Output**: Generated pubmed_abstracts.csv and hw6_pubmed_trend.png

In [ ]:
# HW6: PubMed Literature Mining & Publication Trend

# 1. Author your 3-line Ethics Statement:
# Line 1 (Permission): ...
# Line 2 (Politeness): ...
# Line 3 (Attribution): ...

def execute_hw6_pipeline(query: str, max_records: int, out_csv: Path, out_img: Path) -> pd.DataFrame:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test HW6 — run me. No output means you passed.
out_csv_hw6 = OUT_DIR / "pubmed_abstracts.csv"
out_img_hw6 = OUT_DIR / "hw6_pubmed_trend.png"
df_hw6 = execute_hw6_pipeline("CRISPR Cas9 gene editing", 25, out_csv_hw6, out_img_hw6)

assert out_csv_hw6.exists()
assert out_img_hw6.exists()
assert len(df_hw6) >= 10
assert "pmid" in df_hw6.columns and "year" in df_hw6.columns
print("HW6 passed ✔")

## 6. Summary & Cheat Sheet

In this capstone homework module, you synthesized data extraction, ethical governance, clean data modeling, and publication-quality visual communication across six diverse biological domains. Every automated workflow produced a validated CSV table, an informative figure, and an ethics statement.

### Quick Reference Table

| Task | Visualization Strategy | Output Deliverable | Biological Insight |
| --- | --- | --- | --- |
| HW1: Gene Expression | Grouped Bar + SE Errorbars | gene_expression.csv | Tissue specificity profiling |
| HW2: UniProt Proteins | Histogram + KDE Overlay | protein_info.csv | Protein length & mass distribution |
| HW3: Clinical Trials | Phase Frequency Barplot | clinical_trials.csv | Translational pipeline progression |
| HW4: Taxonomy Hierarchy | Phylum Diversity Barplot | taxonomy.csv | Comparative model organism lineage |
| HW5: Pathway Overlaps | Member Count + Intersections | pathway_genes.csv | Signaling network cross-talk |
| HW6: Literature Mining | Publication Trend Time Series | pubmed_abstracts.csv | Bibliometric discovery velocity |

## 7. Further Reading & Resources

- [Biopython Cookbook](https://biopython.org/DIST/docs/tutorial/Tutorial.html): Comprehensive reference recipes for all Biopython submodules.
- [NIH Scientific Data Management Policies](https://sharing.nih.gov/data-management-and-sharing-policy): Federal standards for biological data curation and reproducibility.
- [Nature Methods Guidelines for Figures](https://www.nature.com/nmeth/for-authors/preparing-your-figures): Author guidance on data visualization and accessibility.